## Data Cleaning

In this notebook we will clean the orders, orderlines and products datasets

In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None) # to increase the width of the columns

In [3]:
# uploading the datasets from local
# orders
path = './data/orders.csv'
orders_original = pd.read_csv(path)

# orderlines
path = './data/orderlines.csv'
orderlines_original = pd.read_csv(path)

# products
path = './data/products.csv'
products_original = pd.read_csv(path)

Creating a copy of the datasets. This ensures that our changes won't effect original DataFrames.

In [4]:
orders_df = orders_original.copy()
orderlines_df = orderlines_original.copy()
products_df = products_original.copy()

### 1.  Exploring using info


We begin the data cleaning by exploring data using .info(). This will tell us about 
* shape of DataFrame
* name of columns
* missing values info (if any)
* datatypes of columns

In [5]:
# orders
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 6.9 MB


* total_paid has 5 missing values
* created_ date should be datetime datatype
* one row is one order

In [6]:
# orderlines
orderlines_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                293983 non-null  int64
 1   id_order          293983 non-null  int64
 2   product_id        293983 non-null  int64
 3   product_quantity  293983 non-null  int64
 4   sku               293983 non-null  str  
 5   unit_price        293983 non-null  str  
 6   date              293983 non-null  str  
dtypes: int64(4), str(3)
memory usage: 15.7 MB


* unit_price should be float datatype
* date should be datetime datatype
* one row is one product in a order

In [7]:
# products
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          19326 non-null  str  
 1   name         19326 non-null  str  
 2   desc         19319 non-null  str  
 3   price        19280 non-null  str  
 4   promo_price  19326 non-null  str  
 5   in_stock     19326 non-null  int64
 6   type         19276 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1.0 MB


* desc has 7 missing values
* price has 46 missing values
* type has 50 missing values
* price should be float datatype
* promo_price should be float data type

### 2. Exploring using sample

In [8]:
# orders
orders_df.sample(10)

,order_id,created_date,total_paid,state
56318,355854,2017-05-15 09:22:18,49.99,Shopping Basket
184611,485018,2018-01-08 08:02:07,4404.58,Shopping Basket
148679,448812,2017-11-27 16:59:55,185.07,Shopping Basket
58032,357569,2017-05-20 15:40:39,0.00,Place Order
166895,467206,2017-12-19 21:16:16,32.99,Completed
167888,468233,2017-12-20 18:37:30,299.00,Shopping Basket
147936,448051,2017-11-27 12:57:12,1993.97,Shopping Basket
196788,497272,2018-01-23 22:23:33,29.99,Shopping Basket
174491,474879,2017-12-28 13:42:10,785.08,Shopping Basket
170561,470933,2017-12-24 21:24:45,23.74,Shopping Basket


In [9]:
# orders
# state has different unique values. We can find the distinct values using unique
print(orders_df['state'].nunique())
orders_df['state'].unique()

5


<StringArray>
['Cancelled', 'Completed', 'Pending', 'Shopping Basket', 'Place Order']
Length: 5, dtype: str

In [10]:
# order_lines
orderlines_df.sample(10)

,id,id_order,product_id,product_quantity,sku,unit_price,date
227152,1542258,482490,0,1,LOG0240,99.99,2018-01-06 15:07:53
52692,1226143,345233,0,1,APP1224,1.328.99,2017-04-11 00:11:49
287763,1641610,523905,0,1,APP2161,179.00,2018-03-09 17:07:24
114242,1347722,397746,0,1,OWC0184,756.58,2017-09-08 10:36:33
109280,1326657,393608,0,1,LAC0129,231.79,2017-08-28 19:28:08
88809,1289407,375800,0,1,QNA0152,454.99,2017-07-11 11:53:18
46486,1215039,339891,0,1,SEA0039,72.99,2017-03-28 02:20:23
264174,1602562,506860,0,1,APP2371,1.675.59,2018-02-06 14:52:42
119050,1356461,402028,0,1,APP1633,870.33,2017-09-19 22:17:35
118651,1355743,401690,0,1,PAC1158,699.18,2017-09-19 09:26:17


orderlines
* product_id shows multiple zero values
* there are multiple decimal point in unit_price

In [11]:
# orderlines - product_id
orderlines_df['product_id'].unique()

array([0])

In [12]:
# orderlines - product_id
# as product_id contains only zero values, we can drop this column
orderlines_df = orderlines_df.drop(columns='product_id')

In [13]:
# orderline - unit_price
((orderlines_df['unit_price'].str.count(r'\.') > 1)|(orderlines_df['unit_price'].str.contains("\d+\.\d{3,}"))).sum()

np.int64(36169)

In [14]:
# products
products_df.sample(10)

,sku,name,desc,price,promo_price,in_stock,type
4700,PAC1038,"Apple iMac 27 ""Core i5 3.3GHz Retina 5K | 32GB | 3TB Fusion",IMac desktop computer 27 inch 5K Retina i5 3.3GHz RAM 32GB 3TB Fusion (MK482Y / A).,3469,2800,0,"5,74E+15"
12478,PAC1418,Synology DS716 + II Pack | 8GB RAM | WD 16TB Network,Synology DS716 + II with 8GB of RAM memory + 16TB (2x8TB) WD Red for Mac and PC,1269.89,11.141.789,0,12175397
17748,PAC2258,DS418play Synology NAS Server | 16GB RAM | 16TB (4x4TB) WD Red,4-bay NAS server to accommodate 4K Ultra HD files,1353.71,11.863.675,0,12175397
4053,APP1387,"Apple iMac 27 ""Core i5 3.3GHz Retina 5K | 8GB | 256GB Flash | R9 M395X 4GB",IMac desktop computer 27 inch 8GB RAM 256GB Retina 5K Flash (MK482Y / A).,2929,27.895.848,0,"5,74E+15"
3632,APP1379,"Apple iMac 27 ""Core i5 3.3GHz Retina 5K | 8GB | 512GB Flash",IMac desktop computer 27 inch 8GB RAM 512GB Retina 5K Flash (MK482Y / A).,2869,27.315.847,0,"5,74E+15"
9456,APP1345,"Apple iMac 27 ""Core i7 Retina 5K 4GHz | 8GB | 256GB Flash",IMac desktop computer 27 inch 8GB RAM 256GB Retina 5K Flash (MK472Y / A).,2809,26.755.847,0,"5,74E+15"
12620,THU0043,"Thule Stróvan Sleeve Case MacBook 13 ""(Air Pro Retina) Gray",Cover with pockets and compartments for MacBook Pro 13-inch MacBook Air 13-inch MacBook Pro 13-inch Retina and iPad,49.95,399.905,0,13835403
8843,PAC1563,"Apple iMac 215 ""Core i5 16GHz | 8GB RAM | 1TB SSD",Desktop computer iMac Core i5 215 inches 16GHz | 8GB RAM | 1TB SSD (MK142Y / A),1999,14.499.902,0,1282
18179,OTT0172,OtterBox Defender iPhone Case 8 Plus / 7 Plus / 6s Plus / 6 Plus,Extra strong triple layer cover for your iPhone successfully support the harsh treatment of daily,59.99,419.894,1,5403
18473,APP2704,"Apple iMac Pro 27 ""18-core Intel Xeon W 23GHz | 64GB | 1TB SSD | Radeon Pro Vega 56",Pro iMac 27 inch screen Retina 5K and Intel Xeon processor W of 23GHz,9339,8.779.005,0,118692158


products
* there are multiple decimal point in price
* there are multiple decimal point in promo_price

In [15]:
# products - price
((products_df['price'].str.count(r'\.') > 1)|(products_df['price'].str.contains("\d+\.\d{3,}"))).sum()

np.int64(596)

In [16]:
# products - promo_price
((products_df['promo_price'].str.count(r'\.') > 1)|(products_df['promo_price'].str.contains("\d+\.\d{3,}"))).sum()

np.int64(18060)

### 3.  Duplicates



We are checking for the duplicate rows using duplicated() method. Afterwards we can delete these rows using drop.duplicates()

In [17]:
# orders
orders_df.duplicated().sum()

np.int64(0)

In [18]:
# orderlines
orderlines_df.duplicated().sum()

np.int64(0)

In [19]:
# products
products_df.duplicated().sum()

np.int64(8746)

In [20]:
# products
# we are deleting the duplicate rows
products_df = products_df.drop_duplicates()

In [21]:
# products
# rechecking to confirm that there are no duplicate rows
products_df.duplicated().sum()

np.int64(0)

### 4. Missing values

In [22]:
# orders - total_paid
# total_paid has 5 missing values. Let's find the percentage using value_counts()
orders_df['total_paid'].isna().value_counts(normalize = True) * 100

total_paid
False    99.997796
True      0.002204
Name: proportion, dtype: float64

In [23]:
# orders - total_paid
# missing values represents only 0.0022 %. As it is very small, we can simply delete the rows
orders_df = orders_df.loc[~orders_df['total_paid'].isna()]

In [24]:
# orders - total_paid
# rechecking to confirm that there are no missing values
orders_df['total_paid'].isna().sum()

np.int64(0)

In [25]:
# orderlines
# there are no missing values in orderlines

In [26]:
# products - desc
# desc has 7 missing values.
products_df.loc[products_df['desc'].isna()]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard drive and NAS",NaN,107,814.659,0,1298
16128,APP1622-A,"Open - Apple Smart Keyboard Pro Keyboard Folio iPad 9.7 """,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,"Open - Kanex USB-C Gigabit Ethernet Adapter MacBook 12 """,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror and Comic Blond,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador T6 Intelligent Wireless (cable),NaN,199.99,1.441.174,0,11905404


In [27]:
# products - desc
# as product names are quite descriptive we can copy them to description column. So that we don't have to delete these rows
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [28]:
# products - desc
# rechecking to confirm that there are no missing values
products_df['desc'].isna().sum()

np.int64(0)

In [29]:
# products - price
# price has 46 missing values. Let's find the percentage using value_counts()
products_df['price'].isna().value_counts(normalize=True) * 100

price
False    99.565217
True      0.434783
Name: proportion, dtype: float64

In [30]:
# products - price
# As the percentage is minute. We can delete these rows using .loc or dropna
products_df = products_df.loc[~products_df['price'].isna()]

In [31]:
# products - price
# rechecking to confirm that there are no missing values
products_df['price'].isna().sum()

np.int64(0)

products - type

Type isn’t an essential piece of data for the analysis and is therefore allowed to carry missing values. The only place it comes in later is as an optional route to category creation, where someone might still choose to drop the rows with missing values, however one can still use name and desc to categorize those rows.


### 5. Datatypes

In [32]:
# orders 
# created_date should be datetime datatype
orders_df['created_date'] = pd.to_datetime(orders_df['created_date'])

In [33]:
# orders
# rechecking to confirm
orders_df.info()

<class 'pandas.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[us]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 8.7 MB


In [34]:
# orderlines
# date should be datetime datatype
orderlines_df['date'] = pd.to_datetime(orderlines_df['date'])

In [35]:
# orderlines
# unit_price has 36169 multidecimal point problem. We can find the percentage using valuecounts
((orderlines_df['unit_price'].str.count(r'\.') > 1)|(orderlines_df['unit_price'].str.contains("\d+\.\d{3,}"))).value_counts(normalize=True)


unit_price
False    0.876969
True     0.123031
Name: proportion, dtype: float64

orderlines
* 12.3% of rows in our orderlines has multiple decimal points.
* We cannot convert these values to float as pandas will produce error due to muldecimal points. Using coerce will turn these values Nan, so that is not recommended
* Due to time constraint we are deleting these rows (might result in loss of large portion of data)
* Each row in orderline represents a product in an order. So if one product has this problem, we have to remove the whole order. We therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [36]:
# orderlines
# creating mask for the multiple decimal point
multi_decimal_mask = ((orderlines_df['unit_price'].str.count(r'\.') > 1)|(orderlines_df['unit_price'].str.contains("\d+\.\d{3,}")))

# getting the affected odrer_ids using boolean mask
corrupted_id_orders = orderlines_df.loc[multi_decimal_mask, 'id_order']

# Negating to get the rows that donot have multiple decimal point
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_id_orders)]

In [37]:
# orderlines
orderlines_df.shape[0]

216250

In [38]:
# orderlines
# now we have 216250 rows to work with.
# converting the unit_price to float
orderlines_df['unit_price'] = pd.to_numeric(orderlines_df['unit_price'])

In [39]:
# orderlines
# rechecking to confirm
orderlines_df.info()

<class 'pandas.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216250 non-null  int64         
 1   id_order          216250 non-null  int64         
 2   product_quantity  216250 non-null  int64         
 3   sku               216250 non-null  str           
 4   unit_price        216250 non-null  float64       
 5   date              216250 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 11.5 MB


In [40]:
# products
# finding percentage of multi decimal problem in price
((products_df['price'].str.count(r'\.') > 1)|(products_df['price'].str.contains("\d+\.\d{3,}"))).value_counts(normalize=True) * 100

price
False    94.854756
True      5.145244
Name: proportion, dtype: float64

In [41]:
# products
# we will delete these 5.14% rows as we want this column to be very trustworthy as we need it to find discount
products_df = products_df.loc[~((products_df['price'].str.count(r'\.') > 1)|(products_df['price'].str.contains("\d+\.\d{3,}")))]

In [42]:
# products
# converting price into float
products_df['price'] = pd.to_numeric(products_df['price'])

In [43]:
# products
# finding percentage of multi decimal problem in promo_price
((products_df['promo_price'].str.count(r'\.') > 1)|(products_df['promo_price'].str.contains("\d+\.\d{3,}"))).value_counts(normalize=True) * 100

promo_price
True     92.393915
False     7.606085
Name: proportion, dtype: float64

In [44]:
# products _ promo_price
# over 92% of the data in this column is corrupt. 
# There's no point deleting all of these rows, then we would barely have a products table. Instead, as it's only this column that appears to be very untrustworthy, we will delete the column.
products_df = products_df.drop(columns='promo_price')

In [45]:
# products
# rechecking to confirm
products_df.info()

<class 'pandas.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   str    
 1   name      9992 non-null   str    
 2   desc      9992 non-null   str    
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 546.4 KB


### 6. Resetting index and saving changes

In [46]:
# orders
orders_df = orders_df.reset_index(drop=True)
orders_cl = orders_df

In [47]:
# orderlines
orderlines_df = orderlines_df.reset_index(drop=True)
orderlines_cl = orderlines_df

In [48]:
# products
products_df = products_df.reset_index(drop=True)
products_cl = products_df

### Downloading new DataFrames

In [49]:
# saving locally
# orders
orders_cl.to_csv('./data/orders_cl.csv', index=False)

# orderlines
orderlines_cl.to_csv('./data/orderlines_cl.csv', index=False)

# products
products_cl.to_csv('./data/products_cl.csv', index=False)